# External NASA-pretrained encoders → Lebanese occurrence heads

Three real IBM/NASA Prithvi checkpoints are adapted using NASA HLS landscape crops and the existing weather/environment tensors. These are image encoders, not pretrained wildfire classifiers. Most encoder weights remain frozen; one arm tunes the original final LayerNorm.

Modern pretraining and reprocessed imagery were unavailable in historical prediction years. Later-year tests were already viewed. **This is a retrospective research comparison and production approval is false.**

See `docs/pretrained-sources.md`, `docs/pretrained-results.md` and `docs/pretrained-statecharts.md`.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from firecast.benchmark_temporal import sha, evaluate
from firecast.benchmark_pretrained import subset

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'outputs/pretrained-transfer'
report = json.loads((OUT / 'results.json').read_text())
frozen = report['selection']
assert sha(OUT / 'selection-frozen.json') == report['selection_sha256']
assert report['production_approved'] is False
print('Validation-selected overall:', frozen['selected_overall'])
print('Validation-selected pretrained arm:', frozen['selected_pretrained'])
print('Fresh independent confirmation:', report['fresh_confirmation'])

Validation-selected overall: prithvi-v2-100m-tl-frozen_encoder
Validation-selected pretrained arm: prithvi-v2-100m-tl-frozen_encoder
Fresh independent confirmation: False


## Acquisition, exclusions and chronological splits

Cloud/cirrus/adjacency/shadow/snow/water/fill/high aerosol and invalid-reflectance pixels fail HLS screening. Keep at least 95% valid pixels and retain the exact raw crop/masks. Missing-image positives remain excluded rather than becoming negatives. Existing fire-mask QA and southern conflict quarantines remain upstream.

Fit 2018–2020 → calibration 2021 → validation 2022 → frozen diagnostic tests 2023–2025 and Jan–Oct 2026. Fourteen-day gaps separate development stages.

In [2]:
pd.DataFrame({**frozen['cohort'], **report['test_cohort']}).T

,original_windows,original_positive_windows,kept_windows,kept_positive_windows,missing_hls_excluded_windows,missing_hls_excluded_positive_windows
train,109567,197,102055,165,7512,32
calibration,35528,98,33041,81,2487,17
validation,35086,32,32591,29,2495,3
lebanon_known_2023_2025,92746,133,87757,113,4989,20
lebanon_known_2026_JanOct,23367,15,22268,14,1099,1


In [3]:
rows = []
for name, val in frozen['validation'].items():
    old = report['results'][name]['lebanon_known_2023_2025']
    recent = report['results'][name]['lebanon_known_2026_JanOct']
    rows.append({'candidate': name, 'validation_AP': val['pr_auc_average_precision'],
                 'known_2023_2025_AP': old['pr_auc_average_precision'],
                 'known_2026_AP': recent['pr_auc_average_precision'],
                 'known_2023_2025_precision': old['precision'], 'known_2023_2025_recall': old['recall']})
pd.DataFrame(rows).set_index('candidate')

,validation_AP,known_2023_2025_AP,known_2026_AP,known_2023_2025_precision,known_2023_2025_recall
candidate,,,,,
seasonal,0.001314,0.002469,0.000979,0.002591,0.752212
weather-logistic,0.001816,0.002979,0.002050,0.003508,0.168142
environment-logistic,0.002530,0.005196,0.003684,0.004956,0.513274
no-image-mlp,0.002720,0.005346,0.003923,0.004713,0.522124
prithvi-v1-100m-frozen_encoder,0.002850,0.009231,0.002708,0.024055,0.061947
prithvi-v1-100m-final_layernorm_tuning,0.002707,0.007925,0.002875,0.027972,0.035398
prithvi-v2-tiny-tl-frozen_encoder,0.002810,0.006692,0.002788,0.017123,0.088496
prithvi-v2-tiny-tl-final_layernorm_tuning,0.002973,0.006723,0.003041,0.013793,0.053097
prithvi-v2-100m-tl-frozen_encoder,0.002994,0.016956,0.002962,0.030303,0.044248


## Verify actual fine-tuning and reproduce a frozen result

This cell uses the shared prediction code with **no labels in the model input**. Labels are used only by the evaluator afterward. The model and threshold were selected on validation, not from this test.

In [4]:
identifier = frozen['selected_pretrained']
record = frozen['artifacts'][identifier]
assert sha(OUT / record['file']) == record['sha256']
artifact = joblib.load(OUT / record['file'])  # only our trusted, hash-verified artifact
name = 'lebanon_known_2023_2025'
with np.load(ROOT / 'outputs/greek-known-benchmark-data' / (name + '.npz')) as arrays:
    data, audit = subset({k: arrays[k] for k in arrays.files}, frozen['common_cells'])
inputs = {k: v for k, v in data.items() if k != 'y'}
probabilities = artifact['model'].predict(inputs)
measured = evaluate(data, probabilities, artifact['threshold'])
assert np.isclose(measured['pr_auc_average_precision'], report['results'][identifier][name]['pr_auc_average_precision'])
for candidate, entry in frozen['artifacts'].items():
    if candidate.endswith('final_layernorm_tuning'):
        assert entry['training']['norm_weight_change_l2'] > 0
    elif candidate.endswith('frozen_encoder'):
        assert entry['training']['norm_weight_change_l2'] == 0
print('Reproduced AP:', measured['pr_auc_average_precision'])
print('Confusion counts TN/FP/FN/TP:', measured['confusion_matrix_tn_fp_fn_tp'])
print('Pretrained parameters updated:', artifact['training'])

Reproduced AP: 0.016956280384046805
Confusion counts TN/FP/FN/TP: [87484, 160, 108, 5]
Pretrained parameters updated: {'seconds': 21.550775654000063, 'fit_windows': 102055, 'positive_windows': 165, 'epochs': 12, 'positive_weight': 10.0, 'seed': 20261115, 'pretrained_norm_tuned': False, 'norm_weight_change_l2': 0.0, 'backbone_trainable_parameters': 0, 'all_trainable_parameters': 28257, 'earlier_transformer_blocks_frozen': True}


## Pipeline statechart

```mermaid
stateDiagram-v2
    [*] --> VerifyPublishedNASAWeights
    VerifyPublishedNASAWeights --> ScreenPastHLS
    ScreenPastHLS --> FrozenTransformerTokens
    FrozenTransformerTokens --> FrozenOrTunedFinalLayerNorm
    FrozenOrTunedFinalLayerNorm --> WeatherAndOccurrenceHead
    WeatherAndOccurrenceHead --> SeparateCalibration
    SeparateCalibration --> ValidationSelection
    ValidationSelection --> FrozenKnownYearEvaluation
    FrozenKnownYearEvaluation --> ResearchOnly
```

No new production approval follows from a better retrospective score. Full attention-block fine-tuning, current spatially representative HLS and an independent issued-weather backtest remain unverified.